# Bài 6(b) — Tập phổ biến bằng FP-Growth

**Võ Thiện Linh — MSSV: 6451071041**

Dùng Groceries Dataset (`groceries.csv`) với `min_sup` từ 0,01 đến 0,1 theo tỉ lệ (1%–10%), tăng 0,01. Thời gian là trung vị của ba lần chạy; không tính thời gian đọc dữ liệu.

## Đọc và kiểm tra dữ liệu

In [ ]:
import csv
import math
import time
from collections import Counter
from pathlib import Path
from statistics import median

import pandas as pd
from IPython.display import display

data_candidates = [Path('groceries.csv'), Path('6451071041_LAB1_Bai6_Trang45') / 'groceries.csv', Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai6_Trang45' / 'groceries.csv']
data_path = next((path for path in data_candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy groceries.csv. Thư mục đang chạy: {Path.cwd()}')
with data_path.open(encoding='utf-8', newline='') as csv_file:
    transactions = [frozenset(item.strip() for item in row if item.strip()) for row in csv.reader(csv_file)]
transaction_count = len(transactions)
item_count = len(set().union(*transactions))
print(f'Đã đọc: {data_path.resolve()}')
print(f'Số giao dịch: {transaction_count:,}; số mặt hàng: {item_count:,}')
assert (transaction_count, item_count) == (9835, 169)
support_levels = [percent / 100 for percent in range(1, 11)]
repeat_count = 3

## Thuật toán FP-Growth
FP-Growth nén giao dịch vào cây FP rồi khai thác các cơ sở mẫu điều kiện đệ quy, không sinh ứng viên như Apriori.

In [ ]:
class FPNode:
    def __init__(self, item, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

def fp_growth(transactions, min_count):
    def mine(weighted_transactions, suffix=frozenset()):
        item_counts = Counter()
        for items, weight in weighted_transactions:
            item_counts.update({item: weight for item in set(items)})
        frequent_items = {item: count for item, count in item_counts.items() if count >= min_count}
        if not frequent_items:
            return {}
        root = FPNode(None)
        headers = {item: [] for item in frequent_items}
        for items, weight in weighted_transactions:
            ordered_items = sorted((item for item in set(items) if item in frequent_items), key=lambda item: (-frequent_items[item], item))
            current = root
            for item in ordered_items:
                child = current.children.get(item)
                if child is None:
                    child = FPNode(item, current)
                    current.children[item] = child
                    headers[item].append(child)
                child.count += weight
                current = child
        patterns = {}
        for item in sorted(headers, key=lambda value: (frequent_items[value], value)):
            pattern = suffix | frozenset([item])
            patterns[pattern] = frequent_items[item]
            conditional_paths = []
            for node in headers[item]:
                path = []
                parent = node.parent
                while parent is not None and parent.item is not None:
                    path.append(parent.item)
                    parent = parent.parent
                if path:
                    conditional_paths.append((path, node.count))
            if conditional_paths:
                patterns.update(mine(conditional_paths, pattern))
        return patterns
    return mine([(transaction, 1) for transaction in transactions])

fp_results = {}
rows = []
for min_sup in support_levels:
    min_count = math.ceil(min_sup * transaction_count)
    elapsed = []
    for _ in range(repeat_count):
        start = time.perf_counter()
        frequent = fp_growth(transactions, min_count)
        elapsed.append(time.perf_counter() - start)
    fp_results[min_sup] = frequent
    rows.append({'min_sup': f'{min_sup:.0%}', 'min_count': min_count, 'Số tập phổ biến': len(frequent), 'Thời gian FP-Growth (giây)': median(elapsed)})
fp_summary = pd.DataFrame(rows)
print('Kết quả và thời gian thực hiện (trung vị 3 lần):')
display(fp_summary)
frequent_itemsets = pd.DataFrame([{'min_sup': f'{s:.0%}', 'tập mặt hàng': ', '.join(sorted(items)), 'kích thước tập': len(items), 'số giao dịch hỗ trợ': count, 'support thực tế': count / transaction_count} for s, results in fp_results.items() for items, count in sorted(results.items(), key=lambda pair: (len(pair[0]), tuple(sorted(pair[0]))))])
print(f'Tổng số dòng tập phổ biến qua các ngưỡng: {len(frequent_itemsets):,}')
display(frequent_itemsets)